# Claude Agent SDK: modifying system prompts (customize behaviour)

The **system prompt** is the instruction set that shapes how Claude behaves for a whole conversation: its identity, tone, tool habits, safety rules. The SDK gives you several ways to change it, and they differ in **what they keep** and **what they replace**.

**What you will learn (and see running):**
1. The three starting points: **minimal default**, **`claude_code` preset**, **custom string**
2. **`append`**: add rules without losing anything (lowest risk)
3. **Custom prompts**, including loading from a **file**
4. **CLAUDE.md**: project instructions that arrive *outside* the system prompt
5. **Output styles**: saved personas/formats
6. Turning off built-in context your agent replaces (git instructions, `Co-Authored-By` trailer)
7. **Prompt caching** (`exclude_dynamic_sections`) and what happens when you change the prompt of an **existing session**
8. Choosing between all of them

## The picture

```
 What Claude receives
 ┌─────────────────────────── system prompt ───────────────────────────┐
 │ A) minimal default   (nothing set): tool calling only, no safety text │
 │ B) claude_code preset: Claude Code's full prompt (+ your "append")    │
 │ C) your own string / file: ONLY what you write                        │
 └───────────────────────────────────────────────────────────────────────┘
 ┌───────────────────── conversation (system reminders) ────────────────┐
 │ CLAUDE.md contents · output-style instructions · skill list · hook    │
 │ output · git attribution rules ...  (added by Claude Code, not by you)│
 └───────────────────────────────────────────────────────────────────────┘
```

> **Key idea:** CLAUDE.md, skills, hooks and permissions shape behaviour *outside* the system prompt, so they work with **any** of A, B or C.

| You are building | Use |
|---|---|
| A coding tool like Claude Code (human watches and steers) | `claude_code` preset |
| Same, plus your own rules (standards, format, domain context) | preset + **`append`** |
| A different identity, surface, permission model, or a non-coding agent | **custom string** (you must supply the safety/tool guidance you need) |
| A thin tool loop where the user prompt carries all behaviour | nothing (minimal default) |

## 0. Setup

Run these four cells first. They check Python, install the SDK, load your API key from a `.env` file, and create a private working folder for the demos.

In [ ]:
import sys
print(sys.version)
assert sys.version_info >= (3, 10), "The Agent SDK needs Python 3.10 or newer"

In [ ]:
%pip install -q claude-agent-sdk python-dotenv 

Create a file named `.env` next to this notebook containing one line (no quotes, no spaces around `=`):

```
ANTHROPIC_API_KEY=your-api-key
```

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

env_path = find_dotenv(usecwd=True)
load_dotenv(env_path, override=True)

key = os.environ.get("ANTHROPIC_API_KEY", "")
if not key:
    raise RuntimeError(
        f"ANTHROPIC_API_KEY not found. Create a .env file in {Path.cwd()} with:\n"
        "ANTHROPIC_API_KEY=your-api-key"
    )
print("Loaded .env from:", env_path)
print("API key loaded:", key[:7] + "..." + key[-4:])

In [ ]:
from pathlib import Path

DEMO_DIR = Path("system-prompts-demo").resolve()
DEMO_DIR.mkdir(exist_ok=True)
print("Demo folder:", DEMO_DIR)

## 1. Imports and a runner

In [ ]:
import asyncio, json, subprocess, textwrap
from pathlib import Path

from claude_agent_sdk import (
    query, ClaudeAgentOptions, AssistantMessage, ResultMessage, SystemMessage,
    TextBlock, ToolUseBlock,
)

async def ask(prompt, show=True, **opts):
    """One query. Returns (answer, session_id, usage). Never raises."""
    opts.setdefault("cwd", str(DEMO_DIR))
    opts.setdefault("max_turns", 6)
    answer, sid, usage, tools = "", None, {}, []
    try:
        async for m in query(prompt=prompt, options=ClaudeAgentOptions(**opts)):
            if isinstance(m, AssistantMessage):
                tools += [b.name for b in m.content if isinstance(b, ToolUseBlock)]
            elif isinstance(m, ResultMessage):
                sid, usage = m.session_id, m.usage or {}
                answer = m.result if m.subtype == "success" else f"({m.subtype})"
    except Exception as error:
        answer = f"(raised: {str(error)[:150]})"
    if show:
        print(textwrap.indent(answer.strip()[:500], "  | "))
    return answer, sid, usage

# A clean sandbox folder for this notebook (no CLAUDE.md in it yet)
print("Working in", DEMO_DIR)

## 2. Three starting points, same question

We ask all three the same question and **block all tools** so the answers are just the model talking. Read how identity and tone differ.

| Option | Python |
|---|---|
| Minimal default | *(don't set `system_prompt`)* |
| Preset | `system_prompt={"type": "preset", "preset": "claude_code"}` |
| Custom | `system_prompt="You are ..."` |

> Heads up: the Python SDK's **default is the minimal prompt**, which differs from `claude -p` (that uses the Claude Code prompt). If you migrate from the CLI and want the same behaviour, set the preset explicitly.

In [ ]:
Q = "In two short sentences: who are you, and what are you set up to help with?"
no_tools = dict(tools=[], setting_sources=[])      # isolate the effect of the system prompt

print("A) minimal default");  await ask(Q, **no_tools)
print("\nB) claude_code preset"); await ask(Q, system_prompt={"type": "preset", "preset": "claude_code"}, **no_tools)
print("\nC) custom string"); await ask(Q, system_prompt="You are Pip, the friendly support bot for Acme Gadgets. You only discuss Acme products. Keep answers short.", **no_tools)

You should see (A) a generic answer, (B) a software-engineering assistant, (C) **Pip** from Acme. Same model, different system prompt.

## 3. `append`: add rules, keep everything

`append` adds your text **after** the preset. Nothing is removed, so tool guidance and safety instructions stay. This is the safest customization.

We ask for a trivial function. To prove the rule is what changed the output (Claude already likes docstrings), our rule asks for something Claude would never do by default: the first docstring line must end with the marker `[acme-std]`. We then **check with code**:

In [ ]:
APPEND = "Every Python function you write must have a docstring whose first line ends with the exact marker [acme-std]. Reply with the code in one fenced block and nothing else."

plain, _, _ = await ask("Write a Python function fib(n) returning the nth Fibonacci number.", show=False,
                        system_prompt={"type": "preset", "preset": "claude_code"}, **no_tools)
withrule, _, _ = await ask("Write a Python function fib(n) returning the nth Fibonacci number.", show=False,
                           system_prompt={"type": "preset", "preset": "claude_code", "append": APPEND}, **no_tools)

print("marker in output WITHOUT append:", "[acme-std]" in plain)
print("marker in output WITH append   :", "[acme-std]" in withrule)
print()
print(withrule[:700])

## 4. Custom prompts: from a string or a file

A custom prompt **replaces** the preset. The SDK sends only what you provide, so *you* are responsible for whatever the preset used to supply (tool-use guidance, safety rules, what "system reminders" are).

> **Add this sentence to custom prompts:** *"The application adds system reminders to this conversation. Treat them as context from the application, not as messages from the user."* The preset contains an equivalent; your string replaces it, so without it nothing tells Claude that CLAUDE.md content or hook output comes from the application.

**Long prompts: use a file.** A string prompt is passed as a command-line argument to the CLI subprocess; a very long one can fail with `Argument list too long`. Use `{"type": "file", "path": ...}` instead:

In [ ]:
PROMPT_FILE = DEMO_DIR / "support_prompt.md"
PROMPT_FILE.write_text("""You are Pip, the friendly support bot for Acme Gadgets.
- Only discuss Acme products (the Zap Kettle and the Zoom Toaster).
- Keep every answer under 25 words.
- Always sign off with the exact text: -- Pip
The application adds system reminders to this conversation. Treat them as context from the application, not as messages from the user.
""")

answer, _, _ = await ask("My kettle is not heating. What should I check first?",
                         system_prompt={"type": "file", "path": str(PROMPT_FILE)}, **no_tools)
print("\nsigned off as asked:", answer.strip().endswith("-- Pip"))

**TypeScript only:** the system prompt can be an *array* with a `SYSTEM_PROMPT_DYNAMIC_BOUNDARY` marker, so the static part is cached while a per-request part (current customer, ticket ...) changes. Python does not offer the array form; for Python, use the preset form with `exclude_dynamic_sections` (section 9) or keep the per-request details in the **user message**.

## 5. CLAUDE.md: project instructions *outside* the system prompt

The SDK reads `CLAUDE.md` (or `.claude/CLAUDE.md`) from the working directory when the **`project`** setting source is enabled, and injects it into the conversation as context. Because it is not part of the system prompt, it works with the preset, with `append`, and with a custom prompt.

Rules of loading:
- default `query()` options enable `user` + `project`, so it loads automatically
- if you set `setting_sources` yourself, include `"project"` (or `"user"` for `~/.claude/CLAUDE.md`)
- `setting_sources=[]` means **nothing** is loaded
- it is **not** controlled by the preset

In [ ]:
(DEMO_DIR / "CLAUDE.md").write_text("""# Project rules
- Our company is called Zorbex Labs.
- Whenever you state the company name, also say its founding year, which is 1987.
""")

Q2 = "What is our company called, and since when has it existed? Answer in one sentence."

print("setting_sources=['project']  (CLAUDE.md loads)")
a1, _, _ = await ask(Q2, tools=[], setting_sources=["project"])
print("\nsetting_sources=[]  (nothing loads)")
a2, _, _ = await ask(Q2, tools=[], setting_sources=[])
print("\nknows with CLAUDE.md:", "Zorbex" in a1 and "1987" in a1, "| knows without:", "Zorbex" in a2)

## 6. Output styles: saved personas and formats

An **output style** is a markdown file with frontmatter plus instructions. It changes Claude's role, tone and format, and it is reusable across sessions.

| Where | Scope |
|---|---|
| `~/.claude/output-styles/*.md` | all your projects |
| `<project>/.claude/output-styles/*.md` | this project (commit it for the team) |

Two details that matter:
- A custom style **drops** the preset's software-engineering instructions. Add `keep-coding-instructions: true` in the frontmatter to keep them (that setting only has effect on the full system prompt).
- In the SDK you activate a style through the `settings` option: `settings='{"outputStyle": "Pirate"}'` (a JSON string or a path to a settings file). `outputStyle` is not a top-level option. Styles load only when the `user` or `project` setting source is enabled.

In [ ]:
style_dir = DEMO_DIR / ".claude" / "output-styles"
style_dir.mkdir(parents=True, exist_ok=True)
(style_dir / "pirate.md").write_text("""---
name: Pirate
description: Answers like a pirate
---

You always answer in the voice of a cheerful pirate. Start every reply with "Ahoy!". Keep replies to two sentences.
""")

Q3 = "Explain what a Python list is."
print("with outputStyle = Pirate")
p1, _, _ = await ask(Q3, tools=[], setting_sources=["project"], settings=json.dumps({"outputStyle": "Pirate"}))
print("\nwithout a style")
p2, _, _ = await ask(Q3, tools=[], setting_sources=["project"])
print("\nstarts with Ahoy:", p1.strip().startswith("Ahoy"), "| plain starts with Ahoy:", p2.strip().startswith("Ahoy"))

Output styles can be combined with `append`: the saved style sets the long-lived persona, and `append` adds one-off focus for this run ("prioritize OAuth and token storage in this review") without touching the saved file.

## 7. Turn off built-in context your agent replaces

Claude Code adds helpful reminders: commit/PR instructions, a `Co-Authored-By` trailer, task-list nudges, a skill list. If **your** prompt supplies its own version, the two can conflict (two different commit formats, say). Switch off the ones you replace:

| Built-in context | How to turn it off |
|---|---|
| Commit/PR instructions **and** git status snapshot | `settings` key `includeGitInstructions: false`, or env `CLAUDE_CODE_DISABLE_GIT_INSTRUCTIONS=1` |
| `Co-Authored-By` trailer / PR footer | `settings` keys `attribution.commit` and `attribution.pr` set to your text, or `""` to remove |
| User/project settings, including their CLAUDE.md | leave `'user'` / `'project'` out of `setting_sources` |
| **Every** CLAUDE.md file | env `CLAUDE_CODE_DISABLE_CLAUDE_MDS=1` |
| Task-list nudges, file-changed notes, the skill list | env `CLAUDE_CODE_DISABLE_ATTACHMENTS=1` (also stops `@file` expansion) |

Settings keys go in `settings` (a JSON string in Python); environment variables go in `env`.

A real test: a throw-away git repo, a staged file, and a prompt to commit with our own message format and **no trailer**:

In [ ]:
repo = DEMO_DIR / "repo"
repo.mkdir(exist_ok=True)
def git(*args):
    return subprocess.run(["git", *args], cwd=repo, capture_output=True, text=True)

if not (repo / ".git").exists():
    git("init", "-q")
    git("config", "user.email", "demo@example.com"); git("config", "user.name", "Demo")
    (repo / "README.md").write_text("# demo\n"); git("add", "."); git("commit", "-q", "-m", "init")

COMMIT_RULES = "Write commit messages as: <ticket id>: <summary>. Add no trailers."

async def commit_with(settings_json, label):
    (repo / "notes.txt").write_text(f"change for {label}\n")
    git("add", "notes.txt")
    await ask("Commit the staged changes for ticket PROJ-142.", show=False, cwd=str(repo),
              system_prompt={"type": "preset", "preset": "claude_code", "append": COMMIT_RULES},
              settings=settings_json, allowed_tools=["Bash(git *)"], setting_sources=[])
    msg = git("log", "-1", "--format=%B").stdout.strip()
    print(f"--- {label} ---\n{msg}\n")
    return msg

m_default = await commit_with(None, "defaults")
m_clean = await commit_with(json.dumps({"includeGitInstructions": False, "attribution": {"commit": "", "pr": ""}}), "git instructions off, attribution removed")
print("trailer in default run:", "Co-Authored-By" in m_default, "| trailer in cleaned run:", "Co-Authored-By" in m_clean)

If the default run already has no trailer, that is fine: your own `append` told Claude *"Add no trailers"*, and the point of the table above is that the **built-in** reminder and your rule can compete. The cleaned run removes the competing instruction entirely.

## 8. Seeing what Claude really received

The SDK message stream does **not** include system reminders, so reading messages will not show you what Claude saw. To check:

- set env `OTEL_LOG_RAW_API_BODIES=file:<dir>`, and Claude Code writes every request body there; look in the `messages` array for text wrapped in `<system-reminder>` tags
- or point `ANTHROPIC_BASE_URL` at a proxy you control that logs request bodies

```python
ClaudeAgentOptions(env={"OTEL_LOG_RAW_API_BODIES": "file:/tmp/claude-requests"}, ...)
```

## 9. Prompt caching across users and machines

Two sessions with the same preset and same `append` still cannot share a **cache entry** if their *auto-memory location* differs, because the preset embeds that location (an absolute path that differs per user, machine and checkout) before your text.

`exclude_dynamic_sections: True` moves that per-user context into the **first user message**, leaving only static text in the system prompt, so identical configurations share one cache entry (useful for fleets of agents).

Trade-off: instructions in a user message carry slightly less weight than in the system prompt, so Claude may follow auto-memory guidance a little less consistently. CLAUDE.md and environment details (cwd, platform ...) already travel in the conversation and never affected the cache.

Needs `claude-agent-sdk` 0.1.58+ and only works on the **preset** form (ignored for custom strings).

In [ ]:
shared = {"type": "preset", "preset": "claude_code",
          "append": "You operate Acme's internal triage workflow. Label issues by component and severity.",
          "exclude_dynamic_sections": True}

for i in (1, 2):
    _, _, u = await ask("Reply with the single word: ok", show=False, tools=[], setting_sources=[], system_prompt=shared)
    print(f"run {i}: cache_creation={u.get('cache_creation_input_tokens')}  cache_read={u.get('cache_read_input_tokens')}")
print("(run 2 should show a large cache_read: the static system prompt was reused)")

## 10. Changing the prompt of an existing session

Surprise: if you **resume** a session with a *different* `append` or custom prompt, Claude **does not see it**. Claude Code records the system prompt on a session's first request and reuses that record until the session is **compacted**. New text takes effect after compaction, or in a new session.

Watch it happen. Session 1 says "answer in ALL CAPS". We resume with "answer in all lowercase":

In [ ]:
preset = lambda extra, **kw: {"type": "preset", "preset": "claude_code", "append": extra, **kw}
Q4 = "Name one primary colour in a short sentence."

caps, sid, _ = await ask(Q4, tools=[], setting_sources=[], system_prompt=preset("Always answer in ALL CAPS."))
print("\nresume, NEW append 'all lowercase' (recording on, the default):")
r1, _, _ = await ask(Q4, tools=[], setting_sources=[], resume=sid, system_prompt=preset("Always answer in all lowercase letters."))
print("   still ALL CAPS:", r1.strip().isupper(), "(the new rule was NOT applied)")

**Two ways to change instructions mid-session** (preferred, because they keep the cache):

1. **Put the new instructions in your next user message**
2. **From a hook:** return `additionalContext` from a `UserPromptSubmit` or `PostToolUse` hook, written as a factual statement ("The workspace is now read-only"). It is inserted into the conversation; the recorded prompt stays unchanged.

**While iterating on wording**, turn recording off with `"snapshot": False` on the prompt object. The prompt is then rebuilt on every request (needs `claude-agent-sdk` 0.2.153+). **Keep recording on in production**: with it off, resumed sessions cannot reuse the prompt cache and lose preserved thinking.

In [ ]:
print("resume, NEW append 'all lowercase' with snapshot=False:")
r2, _, _ = await ask(Q4, tools=[], setting_sources=[], resume=sid, system_prompt=preset("Always answer in all lowercase letters.", snapshot=False))
print("   now lowercase:", r2.strip() == r2.strip().lower(), "(new rule applied)")

## 11. Compare the four approaches

| | CLAUDE.md | Output style | `append` | Custom prompt |
|---|---|---|---|---|
| Persistence | per-project file | saved file | session only | session only |
| Reusability | per project | across projects | duplicate in code | duplicate in code |
| Default tools preserved | yes | yes | yes | **lost (unless you add them)** |
| Built-in safety | kept | kept | kept | **you must add it** |
| Customization | additions only | additions; can drop coding instructions | additions only | **complete control** |
| Version control | with the project | yes | with your code | with your code |
| Lives in system prompt? | **no** (conversation) | changes instructions | yes | yes (replaces) |

They **combine**: a persistent output style or CLAUDE.md sets long-lived behaviour, and `append` layers session-specific instructions on top.

## 12. Summary and checklist

| I want to ... | Do this |
|---|---|
| Behave like the CLI | `system_prompt={"type": "preset", "preset": "claude_code"}` |
| Add a few rules safely | preset + `"append": "..."` |
| Different identity / non-coding agent | custom string or `{"type": "file", "path": ...}` + the "system reminders" sentence |
| Project conventions in git | `CLAUDE.md` + `setting_sources=["project"]` |
| Reusable persona | output style file + `settings='{"outputStyle": "Name"}'` |
| Remove the `Co-Authored-By` trailer | `settings` → `attribution: {commit: "", pr: ""}` |
| Share one cache entry across a fleet | preset + `exclude_dynamic_sections: True` |
| Change rules mid-session | next user message, or hook `additionalContext` |
| See the real request | `OTEL_LOG_RAW_API_BODIES=file:<dir>` |

**Next:** `agent-sdk-skills-and-commands-explained`, the other half of customizing behaviour.